# NetCore TAC Desk: Explore tickets with Pandas

Run the cells in order from the workspace root or the `notebooks` folder. Load the synthetic ticket CSV and parse `created_at` as a UTC datetime so tickets can be counted by the month they arrived.

In [ ]:
from pathlib import Path

import pandas as pd

workspace_root = Path.cwd()
if workspace_root.name == "notebooks":
    workspace_root = workspace_root.parent

csv_path = workspace_root / "data" / "tac_tickets.csv"
raw = pd.read_csv(csv_path, parse_dates=["created_at"])

print(f"Loaded {len(raw):,} tickets across {raw.shape[1]} columns.")
print(f"Creation dates (UTC): {raw['created_at'].min():%Y-%m-%d} to {raw['created_at'].max():%Y-%m-%d}")

## Inspect the data

`head()` previews ticket records; `info()` shows types and non-null counts. `describe()` shows typical and extreme resolution times, `value_counts()` shows the severity mix, and `isna().sum()` shows where reporting data is missing.

In [ ]:
preview_columns = ["ticket_id", "created_at", "severity", "region", "resolution_hours"]
print("First five tickets:")
print(raw[preview_columns].head().to_string(index=False))

print("\nColumn types and non-null counts:")
raw.info()

print("\nTiming summary (minutes for response, hours for resolution):")
print(raw[["first_response_minutes", "resolution_hours"]].describe().round(2))

severity_counts = raw["severity"].value_counts()
print("\nTicket counts by severity:")
print(severity_counts.to_string())

missing_counts = raw.isna().sum()
print("\nMissing values by column:")
print(missing_counts.to_string())

## Select with `loc` and `iloc`

Use `loc` with a ticket ID and named columns to look up a case for a customer follow-up. Use `iloc` with row and column positions to preview a portion of the table regardless of ticket IDs.

In [ ]:
by_ticket_id = raw.set_index("ticket_id")
first_ticket_id = by_ticket_id.index[0]
ticket_lookup = by_ticket_id.loc[[first_ticket_id], ["severity", "region", "resolution_hours"]]
positional_preview = raw.iloc[:3, :4]

print("Ticket lookup by ID:")
print(ticket_lookup.to_string())
print("\nFirst three rows and first four columns by position:")
print(positional_preview.to_string(index=False))

## Filter and rank tickets

Filter to S1 tickets in EMEA for a regional review of critical incidents. Separately, sort **all** tickets by actual resolution time and take the 10 slowest for case review; use ticket ID to break ties.

In [ ]:
s1_emea = raw.loc[(raw["severity"] == "S1") & (raw["region"] == "EMEA"), preview_columns]
slowest_ten = (
    raw.sort_values(["resolution_hours", "ticket_id"], ascending=[False, True])
    .loc[:, preview_columns]
    .head(10)
)

print(f"S1 tickets in EMEA: {len(s1_emea):,} (showing the first five)")
print(s1_emea.head().to_string(index=False))
print("\nTen slowest tickets across all regions and severities:")
print(slowest_ten.to_string(index=False))

## Clean types and fill missing response times

Keep fractional response minutes as nullable floats and set the SLA flag to Boolean. Treat severity, region, and tier as categories. Fill missing response times with the overall mean of recorded times: these are estimates, not evidence that a customer received a response at that time.

In [ ]:
tickets = raw.copy()
fixed_types = {
    "first_response_minutes": "Float64",
    "sla_breached": "boolean",
    "severity": "category",
    "region": "category",
    "customer_tier": "category",
}
tickets = tickets.astype(fixed_types)

response = tickets["first_response_minutes"]
mean_response_minutes = response.mean()
if pd.isna(mean_response_minutes):
    raise ValueError("Mean imputation needs at least one recorded first-response time.")

tickets["first_response_minutes"] = response.fillna(mean_response_minutes)
print(f"Filled {response.isna().sum():,} missing values with the mean: {mean_response_minutes:.2f} min.")
print(f"Missing response times after filling: {tickets['first_response_minutes'].isna().sum()}")
print(tickets[list(fixed_types)].dtypes.to_string())

## Add ETA, error, and long-running flags

Map S1-S4 to fixed ETAs of 4, 24, 48, and 72 hours. Positive `eta_error_hours` means a ticket took longer than estimated and breached its SLA; zero does not breach. Long-running means **over 72 hours** regardless of severity, so an S1 ticket can breach without being long-running.

In [ ]:
fixed_etas = {"S1": 4, "S2": 24, "S3": 48, "S4": 72}
tickets["baseline_eta_hours"] = tickets["severity"].map(fixed_etas).astype("Int64")
required_fields = ["baseline_eta_hours", "resolution_hours", "sla_breached"]
if tickets[required_fields].isna().any().any():
    raise ValueError("SLA verification requires a known severity, resolution time, and stored flag for each ticket.")

tickets["eta_error_hours"] = tickets["resolution_hours"] - tickets["baseline_eta_hours"]
tickets["is_long_running"] = tickets["resolution_hours"] > 72
tickets["sla_breach_recomputed"] = tickets["resolution_hours"] > tickets["baseline_eta_hours"]
sla_mismatches = tickets["sla_breach_recomputed"] != tickets["sla_breached"]

eta_columns = ["ticket_id", "severity", "resolution_hours", "baseline_eta_hours", "eta_error_hours", "is_long_running"]
print(tickets[eta_columns].head().round(2).to_string(index=False))
print(f"Recomputed SLA flags that disagree with the CSV: {sla_mismatches.sum():,}")

## Answer leadership's overall questions

MTTR is the average elapsed time from ticket creation to resolution, not hours of engineer effort. The SLA breach rate is the share that missed its severity's ETA. The long-running count shows how many cases took more than three days to resolve.

In [ ]:
ticket_count = len(tickets)
mttr_hours = tickets["resolution_hours"].mean()
sla_breach_count = tickets["sla_breach_recomputed"].sum()
sla_breach_rate = tickets["sla_breach_recomputed"].mean()
long_running_count = tickets["is_long_running"].sum()

print(f"Tickets: {ticket_count:,}")
print(f"MTTR: {mttr_hours:.2f} hours")
print(f"SLA breaches: {sla_breach_count:,} ({sla_breach_rate:.2%} of tickets)")
print(f"Long-running tickets: {long_running_count:,} ({long_running_count / ticket_count:.2%} of tickets)")

## Group by severity and region

`groupby().agg()` produces one row per severity-region pair. Use counts to compare case volumes and put breach rates in context: a high rate on a handful of tickets is different from the same rate on hundreds. Long-running counts help identify where to review cases that may affect staffing.

In [ ]:
by_severity_region = tickets.groupby(["severity", "region"], observed=True).agg(
    ticket_count=("ticket_id", "size"),
    mttr_hours=("resolution_hours", "mean"),
    sla_breach_rate=("sla_breach_recomputed", "mean"),
    long_running_count=("is_long_running", "sum"),
).sort_index()

print(by_severity_region.to_string(formatters={
    "mttr_hours": "{:.2f}".format,
    "sla_breach_rate": "{:.2%}".format,
}))

## Pivot MTTR by severity and customer tier

Use `pivot_table()` to put severities on rows and customer tiers on columns. Compare tiers within the same severity to see where typical elapsed times might warrant investigation; a difference alone does not show that the tier caused it.

In [ ]:
mttr_pivot = tickets.pivot_table(
    index="severity", columns="customer_tier", values="resolution_hours", aggfunc="mean", observed=True,
)
print("Mean resolution hours by severity and customer tier:")
print(mttr_pivot.round(2).to_string())

## Count tickets created each month

`Grouper` with `freq="MS"` groups creation timestamps by UTC calendar month. The busiest month shows peak ticket intake in this dataset, useful for capacity planning. This is not a count of resolutions or open backlog.

In [ ]:
monthly_volume = tickets.groupby(pd.Grouper(key="created_at", freq="MS")).size().rename("ticket_count")
monthly_display = monthly_volume.rename(index=lambda month: month.strftime("%Y-%m")).rename_axis("month_utc")
print(monthly_display.to_string())

peak_months = monthly_display.index[monthly_display == monthly_display.max()]
print(f"Peak intake: {monthly_display.max():,} tickets in {', '.join(peak_months)} (UTC).")